In [30]:
import asyncio
import websockets
import httpx
import os, random
import json

async def listen_websocket():
    """Listen to WebSocket messages"""
    uri = "ws://localhost/api/v1/activity/ws"
    try:
        async with websockets.connect(uri) as websocket:
            print("Connected to WebSocket server")
            
            while True:
                message = await websocket.recv()
                print("Received:")
                print(json.dumps(json.loads(message), indent=2))
    
    except asyncio.CancelledError:
        print("WebSocket listener cancelled")
    except Exception as e:
        print("WebSocket connection failed:", e)

async def send_post_request():
    """Send POST request with activity data"""
    URL = "http://localhost/api/v1/activity/"
    
    def pick_random_file(folder):
        return os.path.join(folder, random.choice(os.listdir(folder)))
    
    members = ['512929ba-f6ea-4f79-990a-0c5d910b5ccf', '4d818e9b-4d10-4491-af0e-eb68867556d4']
    member = random.choice(members)
    URL += member
    
    skeleton = pick_random_file("E:/ghqut/UTD-MHAD/skeleton")
    inertial = os.path.join("E:/ghqut/UTD-MHAD/inertial", skeleton.replace("skeleton", "inertial"))
    depth = os.path.join("E:/ghqut/UTD-MHAD/depth", skeleton.replace("skeleton", "depth"))
    
    files = {
        "skeleton_file": open(skeleton, "rb"),
        "inertial_file": open(inertial, "rb"),
        "depth_file": open(depth, "rb"),
    }
    
    async with httpx.AsyncClient() as client:
        response = await client.post(URL, files=files)
        print("POST Response:", response)

async def send_requests_periodically():
    """Send POST requests every 2 seconds"""
    await asyncio.sleep(2)  # Wait 2 seconds for WebSocket to connect first
    
    try:
        while True:
            await send_post_request()
            await asyncio.sleep(2)
    except asyncio.CancelledError:
        print("Request sender cancelled")

# Run both tasks concurrently
try:
    await asyncio.gather(
        listen_websocket(),
        send_requests_periodically()
    )
except asyncio.CancelledError:
    print("\nStopped by user")

Connected to WebSocket server
Received:
{
  "predicted_class": 10,
  "predicted_action": "Draw Triangle",
  "confidence": 0.6183975338935852,
  "member": {
    "id": "512929ba-f6ea-4f79-990a-0c5d910b5ccf",
    "name": "Jossin Th"
  }
}
POST Response: <Response [200 OK]>
Received:
{
  "predicted_class": 10,
  "predicted_action": "Draw Triangle",
  "confidence": 0.6183975338935852,
  "member": {
    "id": "512929ba-f6ea-4f79-990a-0c5d910b5ccf",
    "name": "Jossin Th"
  }
}
POST Response: <Response [200 OK]>
Request sender cancelled
WebSocket listener cancelled

Stopped by user
Request sender cancelled
WebSocket listener cancelled

Stopped by user


In [ ]:
!pip install websockets